In [ ]:
# --- Control Flags ---
CREATE_BEV_ANIMATION = False        # Create and display the BEV GT box animation
GENERATE_GT_LABEL_NPZ = True        # Generate the scene-level NPZ files with point labels
GENERATE_ALL_LABELS = True          # If GENERATE_GT_LABEL_NPZ is true, whether to generate labels for all scenes or just one for testing
DISPLAY_K3D_VISUALIZATION = False    # Display the K3D plot for a specific sweep

# --- Imports ---
import os
import sys
import yaml
from nuscenes.nuscenes import NuScenes
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import numpy as np # For K3D example data if NPZs are missing

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('')
# Assuming the notebook is in a 'notebooks' directory at the same level as 'src', 'config', etc.
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules (after adding project root) ---
# For GT Label Generation
from src.data_utils.label_generation import (
    generate_and_save_point_labels_for_scene_hdf5,
    find_instances_in_scene # Used by animation and potentially K3D for GT boxes
)
# For NuScenes sweep data
from src.data_utils.nuscenes_helper import get_scene_sweep_data_sequence
# For BEV Animation of GT Boxes
from src.visualization.animation_helpers import create_synchronized_animation
# For K3D Visualization
from src.visualization.k3d_visualizer import visualize_sweep_k3d # The new K3D function
# For M-Detector (if you want to load its results for K3D)
# from src.core.m_detector.base import MDetector # Not directly needed for GT label gen

# --- Load Configuration ---
try:
    config_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f"Configuration loaded from: {config_path}")
except FileNotFoundError:
    print(f"ERROR: Config file not found at {config_path}. Please check the path.")
    config = {} # Provide a default empty config to avoid further errors if file not found
except Exception as e:
    print(f"Error loading config: {e}")
    config = {}


# --- Initialize NuScenes ---
if 'nuscenes' in config:
    nusc = NuScenes(
        version=config['nuscenes']['version'],
        dataroot=config['nuscenes']['dataroot'],
        verbose=config.get('nuscenes', {}).get('verbose_init', False) # Control initial verbose
    )
    print(f"NuScenes SDK initialized (Version: {config['nuscenes']['version']}).")
else:
    print("ERROR: NuScenes configuration missing in config file. Cannot initialize NuScenes SDK.")
    nusc = None # Set to None if init fails

# --- Select Scene for Demonstration ---
# You can change this index to work with different scenes
target_scene_index = 1 # Example: second scene in the dataset
if nusc and 0 <= target_scene_index < len(nusc.scene):
    my_scene_rec = nusc.scene[target_scene_index]
    print(f"Target Scene for demo: '{my_scene_rec['name']}' (Index: {target_scene_index})")
    print(f"Description: {my_scene_rec['description']}")
else:
    my_scene_rec = None
    print(f"Could not select scene with index {target_scene_index}. Check NuScenes initialization and scene index.")

In [ ]:
if CREATE_BEV_ANIMATION and nusc and my_scene_rec:
    print(f"\n--- Preparing for BEV Animation of GT Boxes for scene: {my_scene_rec['name']} ---")
    
    # Get all LiDAR sweep data dictionaries for the scene (used by animation helper)
    all_sweep_data_for_animation = list(get_scene_sweep_data_sequence(nusc, my_scene_rec['token']))
    
    if not all_sweep_data_for_animation:
        print(f"No LiDAR sweeps found for scene {my_scene_rec['name']}. Skipping BEV animation.")
    else:
        # Find instances in the scene to animate
        # min_annotations=1 ensures instances have at least one GT box to interpolate/extrapolate from
        instances_tokens_for_anim = find_instances_in_scene(nusc, my_scene_rec['token'], min_annotations=1)
        
        # Optional: For faster testing, animate only a few instances
        # instances_tokens_for_anim = instances_tokens_for_anim[:3] 
        
        if not instances_tokens_for_anim:
            print(f"No instances with sufficient annotations found in scene {my_scene_rec['name']}. Skipping BEV animation.")
        else:
            print(f"Found {len(instances_tokens_for_anim)} instances to animate across {len(all_sweep_data_for_animation)} sweeps.")
            
            plt.rcParams['animation.embed_limit'] = 2**128 # Allow large animation embeds
            print("Generating BEV animation. This may take a few minutes...")
            
            # Configuration for the animation from the main config file
            anim_config = config.get('visualization', {}).get('animation_settings', {})
            
            animation_html, animation_object = create_synchronized_animation(
                nusc=nusc,
                instance_tokens=instances_tokens_for_anim,
                all_sweep_data_dicts=all_sweep_data_for_animation,
                scene_name=my_scene_rec['name'],
                interval_ms=anim_config.get('interval_ms', 100), # Default 10 FPS for display
                figsize=tuple(anim_config.get('figsize', (10, 10))),
                point_downsample=anim_config.get('point_downsample', 5),
                save_path=None, # Set to a filepath like "output/animations/gt_boxes_bev.mp4" to save
                save_writer=anim_config.get('save_writer', 'ffmpeg'),
                save_fps=anim_config.get('save_fps', 10), # Corresponds to native LiDAR sweep rate (e.g., 20Hz for 50ms interval)
                save_dpi=anim_config.get('save_dpi', 150)
            )
            print("Done!")
            
            if animation_html:
                print("Preparing BEV animation for display...")
                if isinstance(animation_html, str):
                    display(HTML(animation_html)) # If it's a raw HTML string
                else:
                    display(animation_html)       # If it's already an IPython.display.HTML object
            elif animation_object and anim_config.get('save_path'):
                print(f"BEV Animation saved to: {anim_config.get('save_path')}")
            else:
                print("BEV Animation generation failed or no output produced.")
else:
    if not CREATE_BEV_ANIMATION: print("\nSkipping BEV Animation generation as per flag.")
    if not (nusc and my_scene_rec): print("\nSkipping BEV Animation due to NuScenes/Scene initialization issues.")

In [ ]:
if GENERATE_GT_LABEL_NPZ and nusc:
    print(f"\n--- Preparing for GT Point Label NPZ Generation ---")
    
    # Define the output directory for the GT label NPZ files
    # This path should come from your main configuration file
    gt_labels_output_dir = os.path.join(config.get('nuscenes').get('label_path'))
    os.makedirs(gt_labels_output_dir, exist_ok=True)
    print(f"GT Label NPZ files will be saved in: {gt_labels_output_dir}")

    # --- Option 1: Generate for the currently selected scene (my_scene_rec) ---
    if not GENERATE_ALL_LABELS:
        if my_scene_rec:
            print(f"\nGenerating GT point labels for selected scene: {my_scene_rec['name']}")
            generated_npz_path = generate_and_save_point_labels_for_scene_hdf5(
                nusc, 
                my_scene_rec['token'], 
                gt_labels_output_dir, 
                verbose=True
            )
            if generated_npz_path:
                print(f"GT point labels for scene '{my_scene_rec['name']}' saved to: {generated_npz_path}")
            else:
                print(f"Failed to generate GT point labels for scene '{my_scene_rec['name']}'.")
        else:
            print("No target scene selected (my_scene_rec is None). Skipping single scene GT label generation.")
    else:
        # --- Option 2: Generate for ALL scenes (Uncomment to run) ---
        print("\nStarting GT point label generation for ALL scenes...")
        generated_count = 0
        failed_count = 0
        for scene_idx in range(len(nusc.scene)):
            current_scene = nusc.scene[scene_idx]
            print(f"\n--- Processing scene {scene_idx + 1}/{len(nusc.scene)}: {current_scene['name']} ---")
            npz_path = generate_and_save_point_labels_for_scene_hdf5(
                nusc, 
                current_scene['token'], 
                gt_labels_output_dir, 
                verbose=True
            )
            if npz_path:
                generated_count +=1
            else:
                failed_count +=1
        print(f"\nGT point label generation for ALL scenes complete. Generated: {generated_count}, Failed: {failed_count}.")
else:
    if not GENERATE_GT_LABEL_NPZ: print("\nSkipping GT Label NPZ generation as per flag.")
    if not nusc: print("\nSkipping GT Label NPZ generation due to NuScenes initialization issues.")

In [ ]:
if DISPLAY_K3D_VISUALIZATION and nusc and my_scene_rec:
    print(f"\n--- Preparing for K3D Visualization ---")
    
    # --- Parameters for K3D Visualization ---
    scene_to_visualize_token = my_scene_rec['token']
    scene_to_visualize_name = my_scene_rec['name'] # For printouts
    
    # Select a specific sweep token from the chosen scene
    # For example, the 5th sweep (index 4) if available
    sweeps_in_scene_for_k3d = list(get_scene_sweep_data_sequence(nusc, scene_to_visualize_token))
    sweep_index_to_visualize = 4 # 0-indexed
    
    target_lidar_sd_token_for_k3d = None
    if sweeps_in_scene_for_k3d and 0 <= sweep_index_to_visualize < len(sweeps_in_scene_for_k3d):
        target_lidar_sd_token_for_k3d = sweeps_in_scene_for_k3d[sweep_index_to_visualize]['lidar_sd_token']
        print(f"Selected for K3D: Scene '{scene_to_visualize_name}', Sweep Index {sweep_index_to_visualize}, Token: ...{target_lidar_sd_token_for_k3d[-8:]}")
    else:
        print(f"Could not select sweep at index {sweep_index_to_visualize} for scene '{scene_to_visualize_name}'. Skipping K3D plot.")

    if target_lidar_sd_token_for_k3d:
        # Path to the GT point labels NPZ (generated in the previous cell)
        gt_labels_npz_dir_k3d = os.path.join(PROJECT_ROOT, config.get('paths', {}).get('gt_labels_npz_dir', 'output/gt_point_labels_npz'))
        gt_labels_npz_file_k3d = os.path.join(gt_labels_npz_dir_k3d, f"gt_point_labels_scene_{scene_to_visualize_name}.npz")

        # Path to M-Detector results NPZ (optional, if you want to compare)
        # This path would typically come from your M-Detector run outputs
        mdet_results_dir_k3d = os.path.join(PROJECT_ROOT, config.get('paths', {}).get('mdetector_results_dir', 'output/mdetector_results'))
        mdet_results_npz_file_k3d = os.path.join(mdet_results_dir_k3d, f"mdetector_results_scene_{scene_to_visualize_name}.npz") # Adjust filename if needed
        
        # Check if files exist
        if not os.path.exists(gt_labels_npz_file_k3d):
            print(f"WARNING: GT labels NPZ file not found for K3D: {gt_labels_npz_file_k3d}. GT points/boxes might not be shown.")
            gt_labels_npz_file_k3d = None # Set to None so K3D function handles it
            
        if not os.path.exists(mdet_results_npz_file_k3d):
            print(f"INFO: M-Detector results NPZ file not found for K3D: {mdet_results_npz_file_k3d}. M-Detector points will not be shown.")
            mdet_results_npz_file_k3d = None # Set to None

        k3d_plot_config = config.get('visualization', {}).get('k3d_plot_settings', {})

        print("Generating K3D plot...")
        k3d_plot_object = visualize_sweep_k3d(
            nusc=nusc,
            scene_token=scene_to_visualize_token,
            target_sweep_lidar_sd_token=target_lidar_sd_token_for_k3d,
            gt_labels_npz_path=gt_labels_npz_file_k3d,
            mdet_results_npz_path=mdet_results_npz_file_k3d, # Pass this to show MDet results
            config=config, # Pass the main config, k3d_visualizer will extract its part
            show_gt_boxes=k3d_plot_config.get('show_gt_boxes', True),
            show_gt_points=k3d_plot_config.get('show_gt_points', True),
            show_mdet_points=k3d_plot_config.get('show_mdet_points', True), # Control MDet points visibility
            point_size=k3d_plot_config.get('point_size', 0.05),
            downsample_factor=k3d_plot_config.get('downsample_factor', 1)
        )

        if k3d_plot_object:
            print("Displaying K3D plot...")
            display(k3d_plot_object)
        else:
            print("K3D plot generation failed or no plot object returned.")
else:
    if not DISPLAY_K3D_VISUALIZATION: print("\nSkipping K3D Visualization as per flag.")
    if not (nusc and my_scene_rec): print("\nSkipping K3D Visualization due to NuScenes/Scene initialization issues.")